In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, TimeType,FloatType

In [16]:
from pyspark.sql.functions import col

In [4]:
spark = SparkSession.builder \
    .appName("MiApp") \
    .getOrCreate()    

C:\Users\hever\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
schema1 = StructType([
    StructField('customer_id',
                StringType(), True),
    StructField('full_name',
                StringType(), True),
    StructField('email',
                StringType(), True),
    StructField('phone',
                LongType(), True),
    StructField('city',
                StringType(), True),
     StructField('state',
                StringType(), True),
     StructField('segment',
                StringType(), True),
     StructField('created_at',
                DateType(), True),
     StructField('is_active',
                StringType(), True),
     
    
])

In [6]:
schema4 = StructType([
    StructField('order_id',
                StringType(), True),
    StructField('customer_id',
                StringType(), True),
    StructField('order_status',
                StringType(), True),
    StructField('order_ts',
                DateType(), True),
    StructField('subtotal',
                FloatType(), True),
     StructField('tax',
                FloatType(), True),
     StructField('shipping_fee',
                StringType(), True),
     StructField('total_amount',
                FloatType(), True),
      StructField('currency',
                StringType(), True),
      StructField('source',
                StringType(), True),
])

In [7]:
clientes = spark.read.csv("../data/batch/customers.csv", schema=schema1, header=True)

In [8]:
ordenes = spark.read.csv("../data/batch/orders.csv", schema=schema4,  header=True)

In [9]:
ordenes.show()

+-----------+-----------+------------+----------+---------+--------+------------+------------+--------+-----------+
|   order_id|customer_id|order_status|  order_ts| subtotal|     tax|shipping_fee|total_amount|currency|     source|
+-----------+-----------+------------+----------+---------+--------+------------+------------+--------+-----------+
|ORD00000001| CUST001029|        paid|2025-08-11| 76423.34|12227.73|          49|    88700.07|     MXN|        web|
|ORD00000002| CUST000339|        paid|2026-03-23|  66169.5|10587.12|           0|    76756.62|     MXN| mobile_app|
|ORD00000003| CUST000015|   delivered|2025-11-27| 53389.29| 8542.29|           0|    61931.58|     MXN|marketplace|
|ORD00000004| CUST000274|      packed|2026-02-03| 42101.53| 6736.24|          49|    48886.77|     MXN|marketplace|
|ORD00000005| CUST000232|   cancelled|2025-09-20|100963.26|16154.12|          79|   117196.38|     MXN| mobile_app|
|ORD00000006| CUST000005|   delivered|2026-05-06| 86216.61|13794.66|    

In [10]:
clientes.show()

+-----------+--------------+--------------------+------------+----------------+-----+--------+----------+---------+
|customer_id|     full_name|               email|       phone|            city|state| segment|created_at|is_active|
+-----------+--------------+--------------------+------------+----------------+-----+--------+----------+---------+
| CUST000001|Cliente 000001|cliente000001@exa...|528479902459|Ciudad de México| CDMX|  silver|2025-01-20|     True|
| CUST000002|Cliente 000002|cliente000002@exa...|529963334018|     Guadalajara|  JAL|platinum|2025-03-21|     True|
| CUST000003|Cliente 000003|cliente000003@exa...|521939042955|          Oaxaca|  OAX|  bronze|2025-03-14|     True|
| CUST000004|Cliente 000004|cliente000004@exa...|526241752544|            León|  GTO|    gold|2025-11-22|     True|
| CUST000005|Cliente 000005|cliente000005@exa...|526756332150|     Guadalajara|  JAL|  silver|2025-11-25|     True|
| CUST000006|Cliente 000006|cliente000006@exa...|525693307665|          

In [11]:
join1 = clientes.alias("a").join(ordenes.alias("b"), col("a.customer_id") == col("b.customer_id"), "left")

In [12]:
estado_del_pedido = join1.select("a.customer_id","full_name", "email", "phone", "segment", "is_active", "created_at", "source", "order_status")

In [17]:
estado_del_pedido.select("email").show()

+--------------------+
|               email|
+--------------------+
|cliente000001@exa...|
|cliente000001@exa...|
|cliente000001@exa...|
|cliente000002@exa...|
|cliente000002@exa...|
|cliente000003@exa...|
|cliente000003@exa...|
|cliente000003@exa...|
|cliente000003@exa...|
|cliente000003@exa...|
|cliente000003@exa...|
|cliente000004@exa...|
|cliente000004@exa...|
|cliente000004@exa...|
|cliente000005@exa...|
|cliente000005@exa...|
|cliente000005@exa...|
|cliente000005@exa...|
|cliente000005@exa...|
|cliente000005@exa...|
+--------------------+
only showing top 20 rows


In [18]:
from pyspark.sql.functions import col, regexp_replace

# Supongamos que tu DataFrame se llama 'df' y tu columna 'email'
# Reemplazamos '@example.com' por '@nuevo-dominio.com'

df_actualizado = estado_del_pedido.withColumn(
    "email", 
    regexp_replace(col("email"), r"@example\.com", "@gmail.com")
)

df_actualizado.show()

+-----------+--------------+--------------------+------------+--------+---------+----------+-----------+------------+
|customer_id|     full_name|               email|       phone| segment|is_active|created_at|     source|order_status|
+-----------+--------------+--------------------+------------+--------+---------+----------+-----------+------------+
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|        web|   delivered|
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|call_center|     shipped|
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|marketplace|   delivered|
| CUST000002|Cliente 000002|cliente000002@gma...|529963334018|platinum|     True|2025-03-21|marketplace|   cancelled|
| CUST000002|Cliente 000002|cliente000002@gma...|529963334018|platinum|     True|2025-03-21| mobile_app|     shipped|
| CUST000003|Cliente 000003|cliente000003@gma...|5219390

In [19]:
from pyspark.sql.functions import col, when

# Suponiendo que tu DataFrame original se llama 'df'
df_actualizado = df_actualizado.withColumn(
    "order_status",
    when(col("order_status") == "delivered", "entregado")
    .when(col("order_status") == "shipped", "enviado")
    .when(col("order_status") == "cancelled", "cancelado")
    .when(col("order_status") == "returned", "devuelto")
    .when(col("order_status") == "packed", "empaquetado")
    .when(col("order_status") == "created", "creado")
    .otherwise(col("order_status")) 
)

df_actualizado.show()

+-----------+--------------+--------------------+------------+--------+---------+----------+-----------+------------+
|customer_id|     full_name|               email|       phone| segment|is_active|created_at|     source|order_status|
+-----------+--------------+--------------------+------------+--------+---------+----------+-----------+------------+
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|        web|   entregado|
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|call_center|     enviado|
| CUST000001|Cliente 000001|cliente000001@gma...|528479902459|  silver|     True|2025-01-20|marketplace|   entregado|
| CUST000002|Cliente 000002|cliente000002@gma...|529963334018|platinum|     True|2025-03-21|marketplace|   cancelado|
| CUST000002|Cliente 000002|cliente000002@gma...|529963334018|platinum|     True|2025-03-21| mobile_app|     enviado|
| CUST000003|Cliente 000003|cliente000003@gma...|5219390

In [19]:
estado_del_pedido.write.mode("overwrite").parquet("../data/batch/salida_pedidos.parquet")

Py4JJavaError: An error occurred while calling o89.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 11.0 failed 1 times, most recent failure: Lost task 0.0 in stage 11.0 (TID 11) (DanielNUC executor driver): java.io.IOException: Mkdirs failed to create file:/D:/Documentos Daniel ASUS NUC/Desktop/Proyecto para compartir/data_engineering_streaming_demo/dataatch/salida_pedidos.parquet/_temporary/0/_temporary/attempt_202608212319532333319920202528003_0011_m_000000_11 (exists=false, cwd=file:/D:/Documentos Daniel ASUS NUC/Desktop/Proyecto para compartir/data_engineering_streaming_demo/spark_jobs)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:774)
	at org.apache.hadoop.fs.ChecksumFileSystem.create(ChecksumFileSystem.java:759)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1234)
	at org.apache.hadoop.fs.FileSystem.create(FileSystem.java:1211)
	at org.apache.parquet.hadoop.util.HadoopOutputFile.create(H